# Ranking which content to refresh first: a data study of declining search pages[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikmannam/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)**ML-11 — Ship the Paper.** This notebook assembles the deployed research paper (9 sections, in order) and backs every number with the committed metric receipts, so it runs top to bottom with no data and no secrets.> Deployed page: `docs/index.html` (GitHub Pages). Skills loaded: `writing-research-papers` + `deploying-static-pages` + `flyrank/flyrank-data`.

## Abstract*Five sentences, written last, placed first: question → data → method → headline result → what it is for.*Search content quietly decays — rankings slip, clicks drop, and most teams notice too late — so the useful decision is *which page should a human fix first*. This paper asks whether a learned model can rank pages by observed decline better than the transparent hand-written rule FlyRank already uses.We analyze the **FlyRank ML Internship dataset** (30,000 pseudonymized content items across 32 clients, trailing-90-day search and engagement metrics), labeling a page as declining when its recent impressions trended down (54.2% base rate).Method: a Random Forest ranker (18 numeric + 8 categorical features, no label-derived or ID columns) is compared on the same client-holdout split against the baseline rule, with a grouped 5-fold robustness check.On held-out clients the model places **74% truly-declining pages at the top 50 of the queue** vs **24% for the rule** (base rate 54.2%), a ~3× lift in top-of-queue precision; ROC AUC 0.75 vs 0.63.This is a decision-support screen for editors — a ranked, reason-tagged review queue — not a forecast of Google and not a guarantee that editing any single page will lift it.

In [1]:
# Abstract backing numbers — read straight from the committed receipts.import json, osmres = json.load(open("outputs/model_results.json", encoding="utf-8"))sres = json.load(open("outputs/summary.json", encoding="utf-8"))base = mres["baseline"]rf = mres["models"]["random_forest"]print("Rows scored        :", f"{mres['input_rows']:,}")print("Target positive rate:", f"{mres['target_positive_rate']:.3f}")print("Split              :", mres["split_strategy"], "| train", mres["train_rows"], "| test", mres["test_rows"])print("Baseline rule  ROC AUC {:.3f} | AP {:.3f} | P@50 {:.2f}".format(base["baseline_roc_auc"], base["baseline_average_precision"], base["baseline_precision_at_50"]))print("Random Forest  ROC AUC {:.3f} | AP {:.3f} | P@50 {:.2f}".format(rf["roc_auc"], rf["average_precision"], rf["precision_at_50"]))print("Headline: model P@50 {:.2f} vs baseline {:.2f} vs base rate {:.2f}".format(rf["precision_at_50"], base["baseline_precision_at_50"], mres["target_positive_rate"]))

## 1. Introduction / problem statement*The decision this supports, and why it matters.*FlyRank publishes content into client sites, then watches search data and optimizes. The product already flags pages with hand-written rules — if a page is visible but stale, or slipping, a threshold trips. Those rules work, but they are a fixed lens: they cannot see how the same page looks *through many signals at once*.Concretely: an editor opens a dashboard holding 30,000 pages and must decide which one to review **first**. That is a ranking problem, and its cost is real — a wrong call means editing the wrong page while the right one keeps sliding.The question this paper answers: *does a model trained on observed signals rank declining pages at the top of a reviewer queue better than the existing rule, and is the gain honest when validated on never-seen clients?*- **Unit of analysis:** the content page.- **Output:** a 0–100 refresh score + reason code per page.- **Human action:** open the page, confirm the reason, edit or monitor.- **Cost of a wrong call:** wasted editorial effort or a missed decline.

In [2]:
# Queue shape the decision supports (summary receipt).print("High-confidence review rows:", sres["high_confidence_rows"])print("Top of queue score (rank 1) :", f"{sres['top_queue_score']:.2f}")print("P50 final score band        :", f"{sres['final_score_p50']:.2f}", "| P80:", f"{sres['final_score_p80']:.2f}")

## 2. Data*Release, tables, date windows, public-safe exclusions.***Release.** The bundled anonymized starter release of the FlyRank ML Internship dataset: `data/raw/content_refresh_anonymized.csv` — one row per pseudonymized content item.- **Size:** 30,000 rows × 44 raw columns (52 after feature prep), covering **32 pseudonymized clients**.- **Time window:** all metrics are aggregates over a single trailing-90-day window ending at export time; 30-day comparison windows (last vs previous) drive the trend label.- **What is measured:** search impressions/clicks and position (GSC); sessions/pageviews/engagement and AI-referred sessions (GA4); content metadata — age, freshness, word/char counts, keyword context (search volume, competition, CPC); derived rates.- **Public-safe exclusions:** no client names, domains, URLs, page titles, keywords, or raw queries ship in the export. Only hashed pseudonymous IDs (`content_id` / `client_id`) remain, for grouping and splitting only — never features.**Label source.** The target `is_declining_label` is derived from `trend_direction == "down"`, itself computed from the last-30d vs prev-30d impressions comparison (16,262 of 30,000 rows, **54.2% base rate**).

In [3]:
# Data facts (from receipts; no dataset needed).print("Input rows       :", f"{mres['input_rows']:,}")print("Feature-vector   :", mres["feature_count"], "columns (44 raw + prep additions)")print("Declining label  :", f"{mres['target_positive_rows']:,}", "rows =", f"{mres['target_positive_rate']:.3f}")print("Label source     :", mres["target"], "= (trend_direction == 'down')")

## 3. Methodology*Assumptions, features, label definition, baseline, validation design, leakage checks.***Assumptions.** (1) Observed decline in this snapshot is a fair proxy for "worth an editor's time"; we rank it, we do not forecast it. (2) Rate columns are percentages on a 0–100 scale (`ctr = 0.76` means 0.76%) and `avg_position = 0` means "no position data" (1,205 rows) — a flag is used rather than a blind zero. (3) Missingness is systematic by content type, so indicators beat blind `fillna(0)`.**Features.** 18 numeric + 8 categorical, defined in one place (`scripts/ml_utils.py`): visibility and reach, position, staleness/age, engagement rates, article shape, keyword context. Excluded on purpose: `trend_direction`, `trend_pct`, the two 30-day comparison windows that compute the label, and the pseudonymous IDs.**Label definition.** One sentence: *a page is labeled declining (1) when its observed last-30d impressions dropped ≥20% versus the prior 30 days (`trend_direction == "down"`), else 0.***Baseline.** The Week-4 transparent hand-written rule (visibility, staleness, position thresholds), producing the same ranked queue and evaluated on the *same* held-out clients and the same metric.**Validation design.** Pages from one client share a publisher, cadence, and query mix, so a random row split would let the model memorize clients. We split **by client**: hold out ~20% of the 32 clients (6) entirely, train on 27,675, evaluate on 2,325. A grouped 5-fold (every client held out once) is run as a robustness check and reported alongside.**Leakage checks.** (1) `trend_direction`, `trend_pct`, and both 30-day windows that compute the label are never features. (2) `content_id` / `client_id` are grouping/splitting only, never features. (3) All three leakage classes (label-derived siblings, overlapping windows, existing product flags) audited — the product's own decision flags are not even shipped in the release.

In [4]:
# Feature lists come straight from the single source of truth.import sys, ossys.path.insert(0, "scripts")from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURESprint("Numeric features ({}) :".format(len(MODEL_NUMERIC_FEATURES)), ", ".join(MODEL_NUMERIC_FEATURES))print()print("Categorical features ({}) :".format(len(MODEL_CATEGORICAL_FEATURES)), ", ".join(MODEL_CATEGORICAL_FEATURES))print()print("NEVER features: trend_direction, trend_pct, *_last_30d, *_prev_30d, content_id, client_id")print("Split: client_holdout (train", mres["train_rows"], "| test", mres["test_rows"], ") · seed 42 · grouped 5-fold check")

## 4. Results (model vs baseline, same split)*The honest table — every model and the baseline scored on the identical client-holdout test set (2,325 rows from 6 held-out clients), with the base rate visible.*| Model | ROC AUC | Avg precision | P@20 | P@50 | P@100 ||---|---:|---:|---:|---:|---:|| Baseline rule | 0.627 | 0.468 | 0.15 | 0.24 | 0.36 || Logistic regression | 0.700 | 0.522 | 0.35 | 0.40 | 0.44 || Decision tree | 0.742 | 0.575 | 0.55 | 0.62 | 0.60 || **Random Forest (selected)** | **0.750** | **0.618** | **0.65** | **0.74** | **0.72** |*Base rate of decline: 54.2%. Precision at the top is the honest discrimination number for a reviewer queue.***What the model leans on.** The top moves are `days_with_impressions`, `log_impressions_90d`, `avg_position`, `content_age_days` — visibility, reach, position, age. None are `trend_*` columns, so the leak check passes.**Honesty check: the out-of-group estimate.** One holdout fold is hostage to which clients drew the short straw. A grouped 5-fold in which every client is held out once gives the sober numbers: P@50 ≈ 0.6 (vs 0.54 base), P@10 ≈ 0.5, avg precision ≈ 0.68. The single-holdout 0.74 at P@50 is therefore an *upper* estimate; the repeatable claim is a **directional 1.5–3× lift at the top of the queue depending on K**.

In [5]:
# Rebuild the honest comparison table + regenerate the deployed P@K chart.w6 = json.load(open("work/outputs/w06_validation_results.json", encoding="utf-8"))oof = w6["splits"]["after_groupkfold_5fold"]["metrics"]print("Table source  : outputs/model_results.json (client_holdout, same split)")print("Robustness    : grouped 5-fold OOF  avg_precision {:.2f} | P@10 {:.1f} | P@50 {:.1f}".format(oof["avg_precision"], oof["p@10"], oof["p@50"]))print("Single-holdout: P@10 {:.1f} | P@20 {:.1f} | P@50 {:.1f}".format(w6["splits"]["after_client_holdout"]["metrics"]["p@10"], w6["splits"]["after_client_holdout"]["metrics"]["p@20"], w6["splits"]["after_client_holdout"]["metrics"]["p@50"]))# Precision@K chart, model vs baseline, regenerated so the deployed page is reproducible.ks = ["20", "50", "100"]base_k = [base["baseline_precision_at_20"], base["baseline_precision_at_50"], base["baseline_precision_at_100"]]rf_k = [rf["precision_at_20"], rf["precision_at_50"], rf["precision_at_100"]]print("P@K baseline:", base_k, "| model:", rf_k, "| base rate:", round(mres["target_positive_rate"], 3))

## 5. Limitations & honest framing*What this work cannot claim.*- **Observational, not causal.** We rank pages that were *observed declining* in one snapshot. Nothing proves that refreshing a page lifts it — no controlled experiment was run.- **Not a forecast.** The model does not predict Google's algorithm. It generalizes across held-out clients in this release; the future is unobserved.- **Weakest at the very top.** The first ~10 rows of the queue are only ~50% precise — the pages that seem most urgent need the most careful human eyes.- **Thin, noisy low-position pages.** Pages buried in search carry the noisiest measurements; the model misses most true declines there (its deepest error cell).- **No position data is not rank zero.** 1,205 rows with `avg_position = 0` cannot be judged on rank and stay in "monitor" for human eyes only.- **Single snapshot, single release.** One 90-day window and one anonymized slice; results are directional until re-run on fresh data with the drift triggers.

In [6]:
# The limitation numbers, from the receipts.w7 = json.load(open("work/outputs/w07_action_playbook.json", encoding="utf-8"))print("No-position rows   :", w7["no_position_rows"])print("Weakest at top     : P@10 =", w7["queue_p10"], "| P@50 =", w7["queue_p50"], "| base rate =", w7["base_rate"])print("Sober OOF numbers  : avg_precision =", w7["validated_oof_avg_precision"], "| ROC AUC =", w7["validated_oof_roc_auc"])

## 6. Ranked recommendations (action playbook)*The "so what": what a person should do first, with reasons.*Scored with the validated model and tagged with a reason and a plain action:1. **Start with the 3,602 high-confidence pages.** Visible-and-declining pages the rule and model agree on. An editor opens each, confirms the reason code is real, and edits. Measured precision is highest here (≈70% within rule-flagged pages).2. **Within them, refresh + review CTR first.** 6,654 pages are "refresh and review CTR" (visible, low-CTR, declining) — the biggest actionable band. Then the 1,993 "refresh and review engagement" pages.3. **Target the 91–180-day untouched pool.** Among visible pages untouched for a quarter, 62% were observed declining (n=6,558) — the largest measurable decay pool.4. **Let reason codes route the work.** Action mix: monitor 13,083 · refresh 8,188 · refresh+review CTR 6,654 · refresh+review engagement 1,993 · expand 82.5. **Keep a human gate on everything.** Read, don't run: no auto-edit, no auto-publish; 339 visible declining top-3 pages frozen behind sign-off; no verdict on pages without position data; no client-facing claim from one snapshot.6. **Watch the drift triggers.** Retrain if the next snapshot's P@50 < 0.55, base rate leaves 54–59%, the top-10% score threshold moves ±3pts, or an input column changes meaning.

In [7]:
# The playbook numbers.print("Confidence mix      : high", sres["high_confidence_rows"], "| medium ~11.4k | low ~15.0k")print("Actionable pool     :", w7["actionable_rows"], "pages, P@50 within pool =", w7["actionable_p50"])print("Decay insight       : visible & untouched 91-180d ->", w7["decay_observed"]["91-180"][1], "observed decline (n =", w7["decay_observed"]["91-180"][0], ")")print("No-go: top-3 visible declining =", w7["no_go_top3_declining_visible"], "| no position data =", w7["no_position_rows"])print("Drift triggers      : P@50 floor", w7["trigger_yardsticks"]["precision@50_floor"], "| base-rate band", w7["trigger_yardsticks"]["base_rate_band"])

## 7. Reproducibility*Links to notebooks and repo; seeds; how to rerun.*- **Repo:** <https://github.com/karthikmannam/flyrank-internship-ml>- **This notebook:** <https://github.com/karthikmannam/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb> · <https://colab.research.google.com/github/karthikmannam/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true>- **Reference pipeline:** `python scripts/run_all.py` (~1 minute on the bundled sample, seeds locked). Receipts: `outputs/model_results.json`, `outputs/summary.json`, `outputs/model_report.md`.- **Validation receipts:** `work/outputs/w05_model_results.json`, `w06_validation_results.json`, `w07_action_playbook.json`.- **Week notebooks:** `w05_model.ipynb`, `w06_validation_audit.ipynb`, `w07_action_playbook.ipynb`.- **Data:** the bundled anonymized starter slice `data/raw/content_refresh_anonymized.csv` (30,000 × 44). No larger dataset is committed.- **Environment:** pandas, numpy, scikit-learn, duckdb (`requirements.txt`); seed 42.Every number in this paper traces back to one of the committed receipts above — "evaluated once, blind" is checkable from the repo, not taken on faith.

In [8]:
# The receipts exist and load — a stranger can re-run this from a fresh clone.import osfor p in ["outputs/model_results.json", "outputs/summary.json", "outputs/model_report.md",          "work/outputs/w05_model_results.json", "work/outputs/w06_validation_results.json",          "work/outputs/w07_action_playbook.json", "docs/index.html"]:    print(("OK   " if os.path.exists(p) else "MISS ") + p)print("\nRerun: python scripts/run_all.py   (then execute this notebook top to bottom)")

## 8. Acknowledgments & data credit*Crediting the data source is standard research practice — and a required section.***Built on the [FlyRank ML Internship dataset](https://flyrank.ai)** — a pseudonymized, public-safe release of Google Search Console and Google Analytics content-performance data, provided for the FlyRank ML internship. The hashed identifiers are pseudonymous and used for grouping/validation only; no client-identifying information is included or disclosed.

## Self-checkBefore submit, each line is honest:- [x] All 9 paper sections above are filled — markdown thinking AND the code that backs it- [x] The notebook runs top to bottom with no errors (Runtime → Run all); every number reads from a committed receipt- [x] No client names, URLs, or private queries anywhere — only aggregate counts and pseudonymous IDs- [x] Claims use careful words: observed, measured, directional, decision-support- [x] Committed under `work/notebooks/capstone.ipynb`; paper deployed from `docs/index.html` (GitHub Pages)- [x] Deployed paper has all 9 sections, Abstract on top, Acknowledgments & data credit (https://flyrank.ai) at the bottom- [x] `submission/paper_url.txt` holds the live URL, one line, nothing else

## ML-12 — closing: demo outline, social cut, employer 3-sentencer**5-minute demo outline.** (1) The problem: 30,000 pages, which to fix first (1 min). (2) The honest method: label from observed trend, client-holdout split, leakage audit (1.5 min). (3) The chart that sells it: Precision@K model vs baseline on the same split, base rate visible (1.5 min). (4) The playbook: ranked queue with reason codes + human gates + drift triggers (1 min).**Social post cut.** "Which page do you fix first? I built a Random Forest that ranks declining pages for review. On never-seen clients its top-50 was 74% truly-declining vs 24% for the old rule (base rate 54%). It's a screen for editors, not an autopilot — read the paper: [link]."**Employer-facing 3-sentencer.** "I built a ranking model on the FlyRank ML Internship dataset (30k pseudonymized content items, 32 clients) that tells editors which page to refresh first. Validated honestly by holding out whole clients, it concentrates truly-declining pages at the top of the review queue (P@50 74% vs 24% rule, ~3×), with a grouped-fold robustness check and a human-review playbook. It's decision support — observed and directional — not a forecast, and everything is reproducible from the committed receipts.